## NIS2 preprocessing

Pipeline note:
- canonical classifier for technical requirements: `all-mpnet-base-v2`
- semantic matching is implemented in Phase 3
- MiniLM and SVO-based matching below are retained as baselines / exploratory experiments


In [ ]:
import requests
import subprocess

#1. Download the NIS2 directive HTML page
url = "https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=CELEX:32022L2555"
filename = "nis2_directive.html"
headers = {"User-Agent": "Mozilla/5.0"}

try:
    response = requests.get(url, headers=headers, allow_redirects=True, timeout=30)
    if response.status_code == 200 and "<!DOCTYPE" in response.text[:100].upper():
        with open(filename, "w", encoding="utf-8") as f:
            f.write(response.text)
        print(f"✅ HTML successfully downloaded via requests as '{filename}'")
    else:
        raise Exception(f"Unexpected status {response.status_code}")
except Exception as e:
    print("⚠️ requests method failed:", e)
    # Fallback to curl
    subprocess.run(["curl", "-L", url, "-o", filename], check=True)
    print(f"✅ HTML successfully downloaded via curl as '{filename}'")

#2. Save the downloaded file
#from google.colab import files
#files.download(filename)

📌 2. Clean and segment the NIS2 text into individual requirement units

In [ ]:
import re
import pandas as pd
from bs4 import BeautifulSoup

# 1. Load the downloaded HTML
with open("nis2_directive.html", "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")

# 2. Find all subdivision divs whose id starts with "art_"
divs = soup.find_all("div", class_="eli-subdivision", id=re.compile(r"^art_"))

records = []
for div in divs:
    raw_id = div["id"]              # e.g. "art_21_2_a"
    parts = raw_id.split("_")[1:]   # ['21','2','a']
    # Build a Req ID string: "Article 21(2)(a)"
    req_id = "Article " + parts[0] + "".join(f"({p})" for p in parts[1:])
    # Extract the text inside this subdivision
    text = div.get_text(separator=" ", strip=True)
    records.append({"Req ID": req_id, "Original Text": text})

# 3. Create DataFrame and save
df_nis2 = pd.DataFrame(records)
df_nis2.to_csv("nis2_requirements_html.csv", index=False)
print(f"✅ Parsed {len(df_nis2)} requirement units into 'nis2_requirements_html.csv'")

📌 3. Clean and normalize NIS2 requirement text


In [ ]:
import re
import pandas as pd

# 1. Load the parsed requirements CSV
df_nis2 = pd.read_csv("nis2_requirements_html.csv")

# 2. Define a cleaning function
def clean_requirement_text(text):
    """
    Remove duplicated heading, numeric lists, and normalize whitespace/punctuation.
    """
    # Remove leading "Article X(…)…" if present
    text = re.sub(r'^Article\s+\d+(?:\(\d+\))*(?:\([a-z]\))?\s*', '', text)
    # Remove paragraph numbers like "1.", "2." at the start
    text = re.sub(r'\b\d+\.\s*', '', text)
    # Normalize spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Ensure ending with a period
    if not text.endswith('.'):
        text += '.'
    return text

# 3. Apply the function
df_nis2['Cleaned Requirement'] = df_nis2['Original Text'].apply(clean_requirement_text)

# 4. Preview the first 10 rows
df_nis2[['Req ID', 'Original Text', 'Cleaned Requirement']].head(10)

# 5. Save cleaned requirements
output_file = "nis2_requirements_cleaned.csv"
df_nis2.to_csv(output_file, index=False)
print(f"✅ Saved cleaned requirements to '{output_file}'")

📌 4a. Baseline classifier: remove non-technical requirements with MiniLM-L6


In [ ]:
# 1. Load model
from sentence_transformers import SentenceTransformer, util
import pandas as pd

model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

# 2. Reference examples
reference_technical = "Operators shall implement logging systems to record security-relevant events."
reference_non_technical = "This Directive is addressed to the Member States."

ref_emb_tech = model.encode(reference_technical, convert_to_tensor=True)
ref_emb_non = model.encode(reference_non_technical, convert_to_tensor=True)

# 3. Load the NIS2 cleaned file
input_file = "nis2_requirements_cleaned.csv"
df = pd.read_csv(input_file)

# 4. Compute prediction per article based on semantic similarity
predictions = []
for text in df["Cleaned Requirement"]:
    emb = model.encode(text, convert_to_tensor=True)
    sim_tech = util.cos_sim(emb, ref_emb_tech).item()
    sim_non = util.cos_sim(emb, ref_emb_non).item()
    label = "technical" if sim_tech > sim_non else "non-technical"
    predictions.append({
        "Category": label,
        "Technical_Score": round(sim_tech, 4),
        "NonTechnical_Score": round(sim_non, 4)
    })

# 5. Save and export result
df_pred = pd.concat([df, pd.DataFrame(predictions)], axis=1)
df_pred.to_csv("nis2_classified_MiniLM-L6.csv", index=False)
print("\n✅ Classification completed and saved to 'nis2_classified_MiniLM-L6.csv'")

# 6. Preview the classified article
print(df_pred[["Req ID", "Category", "Technical_Score", "NonTechnical_Score"]].head(10))

In [ ]:
# Filter technical articles from the MiniLM baseline
df_technical = df_pred[df_pred["Category"] == "technical"]

# Save with an explicit baseline name to avoid confusion with the canonical MPNet output
df_technical.to_csv("nis2_only_technical_minilm.csv", index=False)
print("✅ Saved MiniLM baseline output as 'nis2_only_technical_minilm.csv'.")


📌 4b. Canonical classifier: remove non-technical requirements with MPNet


In [ ]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd

# 1. Load model
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# 2. Reference examples
technical_refs = [
    "Systems must enforce multi-factor authentication and restrict privileged access.",
    "Logging of security events must be enabled and monitored.",
    "Vulnerability management processes shall ensure timely patching.",
    "Data at rest and in transit must be encrypted using strong cryptographic protocols."
]

non_technical_refs = [
    "This directive sets out obligations for Member States to coordinate their national strategies.",
    "Authorities shall report annually to the Commission on implementation progress.",
    "Penalties shall be imposed for non-compliance with regulatory measures.",
    "Member States shall designate national competent authorities responsible for oversight."
]


# 3. Compute mean of the embedding per class
emb_tech = model.encode(technical_refs, convert_to_tensor=True).mean(dim=0)
emb_non = model.encode(non_technical_refs, convert_to_tensor=True).mean(dim=0)

# 4. Load the NIS2 cleaned file
df = pd.read_csv("nis2_requirements_cleaned.csv")

# 5. Compute prediction per article based on semantic similarity
predictions = []
for text in df["Cleaned Requirement"]:
    emb = model.encode(text, convert_to_tensor=True)
    sim_tech = util.cos_sim(emb, emb_tech).item()
    sim_non = util.cos_sim(emb, emb_non).item()
    label = "technical" if sim_tech > sim_non else "non-technical"
    predictions.append({
        "Category": label,
        "Technical_Score": round(sim_tech, 4),
        "NonTechnical_Score": round(sim_non, 4)
    })

# 6. Save and export result
df_pred = pd.concat([df, pd.DataFrame(predictions)], axis=1)
df_pred.to_csv("nis2_classified_mpnet.csv", index=False)
print("\n✅ Classification completed and saved to 'nis2_classified_mpnet.csv'")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Carica il file con classificazione completa
df = pd.read_csv("nis2_classified_mpnet.csv")

# 2. Calcola la differenza tra i punteggi
df["Delta"] = abs(df["Technical_Score"] - df["NonTechnical_Score"])

# 3. Istogramma: distribuzione dei punteggi
plt.figure(figsize=(10, 6))
plt.hist(df["Technical_Score"], bins=20, alpha=0.7, label="Technical Similarity", color='steelblue')
plt.hist(df["NonTechnical_Score"], bins=20, alpha=0.7, label="Non-Technical Similarity", color='salmon')
plt.axvline(df["Technical_Score"].mean(), color='blue', linestyle='--', label="Avg Technical")
plt.axvline(df["NonTechnical_Score"].mean(), color='red', linestyle='--', label="Avg Non-Technical")
plt.title("📊 Distribution of Similarity Scores")
plt.xlabel("Cosine Similarity")
plt.ylabel("Number of Articles")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 4. Scatterplot: confronto tecnico vs non tecnico
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="Technical_Score",
    y="NonTechnical_Score",
    hue="Category",
    palette={"technical": "blue", "non-technical": "red"},
    style="Category"
)
plt.title("📈 Technical vs Non-Technical Similarity")
plt.xlabel("Technical Similarity")
plt.ylabel("Non-Technical Similarity")
plt.grid(True)
plt.tight_layout()
plt.show()

# 5. Barplot: conteggio articoli per categoria
plt.figure(figsize=(6, 4))
sns.countplot(x="Category", data=df, palette={"technical": "steelblue", "non-technical": "lightcoral"})
plt.title("📌 Number of Articles by Classification")
plt.xlabel("Category")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Filter technical articles from the canonical MPNet classifier
df_technical = df_pred[df_pred["Category"] == "technical"]

# Save the official input for Phase 3 semantic matching
df_technical.to_csv("nis2_only_technical_mpnet.csv", index=False)
print("✅ Saved canonical technical requirements as 'nis2_only_technical_mpnet.csv'.")


📌 5. Baseline feature extraction: Legal-BERT + spaCy SVO features


In [ ]:
# 🧠 Install dependencies
!pip install -q spacy[transformers] transformers

In [ ]:
# 📌 4 (hybrid): SVO extraction + BERT-EURLEX embeddings

import spacy
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd

# 2. Load spaCy small model for parsing
nlp = spacy.load("en_core_web_sm")

# 3. Load BERT-EURLEX tokenizer + model
tokenizer = AutoTokenizer.from_pretrained("nlpaueb/bert-base-uncased-eurlex")
model = AutoModel.from_pretrained("nlpaueb/bert-base-uncased-eurlex")

# 4. Define extraction function with embedding
def extract_svo_and_embedding(text):
    # SVO via spaCy
    doc = nlp(text)
    subjects   = [tok.text for tok in doc if tok.dep_ in ("nsubj","nsubjpass")]
    if not subjects and doc and doc[0].pos_ == "VERB":
        subjects = ["Member States"]
    verbs      = [tok.lemma_ for tok in doc if tok.pos_ == "VERB"]
    objects    = [tok.text for tok in doc if tok.dep_ in ("dobj","pobj","dative","attr")]
    noun_chunks= [chunk.text for chunk in doc.noun_chunks]

    # Embedding via BERT-EURLEX CLS token
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        emb = model(**inputs).last_hidden_state[:,0,:].squeeze().numpy()

    return {
        "Subjects": subjects,
        "Verbs": verbs,
        "Objects": objects,
        "Noun Chunks": noun_chunks,
        "Embedding": emb.tolist()
    }

# 5. Load cleaned NIS2 requirements
df_nis2 = pd.read_csv("nis2_requirements_cleaned.csv")

# 6. Apply to all requirements
results = df_nis2["Cleaned Requirement"].apply(extract_svo_and_embedding).apply(pd.Series)

# 7. Merge and save
df_out = pd.concat([df_nis2, results], axis=1)
df_out.to_csv("nis2_requirements_svo_embedding.csv", index=False)
print("✅ Saved SVO+embedding to 'nis2_requirements_svo_embedding.csv'")

📌 6. Tag NIS2 requirements with Control Family


In [ ]:
# 📌 2.6: Tag NIS2 requirements with Control Family

import pandas as pd
import re

# 1. Load the SVO+embedding file (ma puoi anche usare il CSV prima dell'embedding)
df = pd.read_csv("nis2_requirements_svo_embedding.csv")

# 2. Define keyword→family mapping
family_map = {
    'Access Control':     ['access', 'authentication', 'authorization', 'account'],
    'Incident Response':  ['incident', 'response', 'notification', 'reporting'],
    'Risk Assessment':    ['risk', 'assessment', 'management'],
    'Supply Chain Risk Management': ['supply chain'],
    'Configuration Management': ['configuration', 'config', 'baseline', 'inventory'],
    'System and Communications Protection': ['encrypt', 'encryption', 'communication', 'vpn', 'tunnel'],
    'System and Information Integrity': ['integrity', 'patch', 'update', 'vulnerability'],
    'Contingency Planning': ['continuity', 'backup', 'recovery', 'resilience'],
    'Audit and Accountability': ['audit', 'log', 'monitor', 'accountability']
    # aggiungi altre famiglie se serve
}

def tag_family(text):
    text_lower = text.lower()
    for family, keywords in family_map.items():
        if any(kw in text_lower for kw in keywords):
            return family
    return 'Other'

# 3. Apply tagging
df['Control Family'] = df['Cleaned Requirement'].apply(tag_family)

# 4. Preview assignment
df[['Req ID', 'Control Family']].drop_duplicates().head(10)

# 5. (Opzionale) Save final CSV for matching
df.to_csv("nis2_requirements_svo_with_family.csv", index=False)
print("✅ Saved tagged requirements as 'nis2_requirements_svo_with_family.csv'")

📌 7. Baseline experiment: SVO-based matching


In [ ]:
import pandas as pd
import ast
from google.colab import files

# 1. Carica i dataset NIS2 e NIST già arricchiti con SVO e Control Family
df_nis2 = pd.read_csv("nis2_requirements_svo_with_family.csv")
df_nist = pd.read_csv("nist_controls_svo_v2_with_family.csv")

# 2. Ripristina le liste Python
for df in (df_nis2, df_nist):
    for col in ['Verbs', 'Objects']:
        df[col] = df[col].apply(ast.literal_eval)

# 3. Pre-raggruppa i controlli NIST per famiglia
nist_by_family = {
    fam: group
    for fam, group in df_nist.groupby("Control Family")
}

# 4. Funzione di matching SVO
def match_svo(req_row, top_k=3):
    fam = req_row["Control Family"]
    candidates = nist_by_family.get(fam, pd.DataFrame())
    req_verbs = set(req_row["Verbs"])
    req_objs  = set(req_row["Objects"])
    scores = []
    for _, ctl in candidates.iterrows():
        ctl_verbs = set(ctl["Verbs"])
        ctl_objs  = set(ctl["Objects"])
        score = len(req_verbs & ctl_verbs) + len(req_objs & ctl_objs)
        if score > 0:
            scores.append((ctl["Control ID"], score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

# 5. Applica il matching a ogni requisito
df_nis2["Top SVO Matches"] = df_nis2.apply(match_svo, axis=1)

# 6. Esporta i risultati in un CSV
output_filename = "nis2_nist_svo_matching.csv"
df_nis2[["Req ID", "Control Family", "Top SVO Matches"]].to_csv(output_filename, index=False)
print(f"✅ Saved matching results to '{output_filename}'")


# 7. Scarica il file
files.download(output_filename)


📌 8. Baseline refinement: weighted SVO matching


In [ ]:
import pandas as pd
import ast
import matplotlib.pyplot as plt

# 1. Carica i due dataset (assicurati che i CSV stiano nella stessa cartella del notebook)
df_nis2 = pd.read_csv("nis2_requirements_svo_with_family.csv")
df_nist = pd.read_csv("nist_controls_svo_v2_with_family.csv")

# 2. Ripristina le liste da stringhe
for df in (df_nis2, df_nist):
    for col in ['Verbs','Objects','Noun Chunks']:
        df[col] = df[col].apply(ast.literal_eval)

# 3. Raggruppa i controlli NIST per Control Family
nist_by_family = {fam: grp for fam, grp in df_nist.groupby("Control Family")}

# 4. Funzione di matching pesato (verb=1, obj=2, chunk=1)
def match_weighted(req, top_k=3, w_v=1, w_o=2, w_c=1):
    fam = req["Control Family"]
    cand = nist_by_family.get(fam, pd.DataFrame())
    rv, ro, rc = set(req["Verbs"]), set(req["Objects"]), set(req["Noun Chunks"])
    scores = []
    for _, ctl in cand.iterrows():
        cv, co, cc = set(ctl["Verbs"]), set(ctl["Objects"]), set(ctl["Noun Chunks"])
        score = len(rv&cv)*w_v + len(ro&co)*w_o + len(rc&cc)*w_c
        if score>0:
            scores.append((ctl["Control ID"], score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

# 5. Calcola i Top 3 weighted matches
df_nis2["Weighted Matches"] = df_nis2.apply(match_weighted, axis=1)

# 6. Copertura: percentuale di requisiti con almeno un match ≥ soglia
threshold = 2
has_match = df_nis2["Weighted Matches"].apply(lambda L: any(s>=threshold for _,s in L))
print(f"Coverage (score ≥ {threshold}): {has_match.mean()*100:.1f}% ({has_match.sum()}/{len(df_nis2)})\n")

# 7. Preview e istogramma dei punteggi top
print(df_nis2[["Req ID","Control Family","Weighted Matches"]].head(10).to_string(index=False))

top_scores = [L[0][1] if L else 0 for L in df_nis2["Weighted Matches"]]
plt.hist(top_scores, bins=range(0, max(top_scores)+2), align="left")
plt.title("Distribuzione dei Top Weighted Match Scores")
plt.xlabel("Score")
plt.ylabel("Numero di requisiti")
plt.tight_layout()
plt.show()

# 8. Esporta i risultati
df_nis2[["Req ID","Control Family","Weighted Matches"]].to_csv(
    "nis2_nist_weighted_svo_matching.csv", index=False
)

📌 9. Baseline diagnostics


In [ ]:
import pandas as pd
import ast

# 1. Ricarica i dati con i match pesati
df = pd.read_csv("nis2_nist_weighted_svo_matching.csv")
df["Weighted Matches"] = df["Weighted Matches"].apply(ast.literal_eval)

# 2. Crea colonna “Has Match ≥2”
df["HasMatch"] = df["Weighted Matches"].apply(lambda L: any(score >= 2 for _,score in L))

# 3. Calcola copertura per famiglia
coverage_by_family = (
    df.groupby("Control Family")["HasMatch"]
      .agg(["sum","count"])
      .assign(Coverage=lambda x: (x["sum"]/x["count"]*100).round(1))
      .reset_index()
      .rename(columns={"sum":"Matched","count":"Total"})
)

coverage_by_family

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.bar(coverage_by_family["Control Family"], coverage_by_family["Coverage"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Coverage (%)")
plt.title("Coverage of Weighted SVO Matches by Control Family")
plt.tight_layout()
plt.show()

📌 10. Baseline plots


In [ ]:
# 📌 2.9: Load data, compute metrics and generate plots with matplotlib

import pandas as pd
import ast
import matplotlib.pyplot as plt
from collections import Counter

# 1. Load matching results and parse the list column
df = pd.read_csv("nis2_nist_weighted_svo_matching.csv")
df['Weighted Matches'] = df['Weighted Matches'].apply(ast.literal_eval)

# 2. Compute derived metrics
df['Num Matches'] = df['Weighted Matches'].apply(len)
df['Top Score']   = df['Weighted Matches'].apply(lambda L: L[0][1] if L else 0)
df['Has Match']   = df['Num Matches'] > 0

# --- Plot 1: Pie chart of matched vs unmatched requirements ---
plt.figure()
sizes = [df['Has Match'].sum(), (~df['Has Match']).sum()]
labels = ['Matched', 'Unmatched']
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140)
plt.title('Matched vs Unmatched Requirements')
plt.tight_layout()
plt.show()

# --- Plot 2: Histogram of number of matches per requirement ---
plt.figure()
plt.hist(df['Num Matches'], bins=range(0, df['Num Matches'].max()+2), align='left')
plt.title('Distribution of Number of Matches per Requirement')
plt.xlabel('Number of Matches')
plt.ylabel('Count of Requirements')
plt.tight_layout()
plt.show()

# --- Plot 3: Histogram of top match score per requirement ---
plt.figure()
plt.hist(df['Top Score'], bins=range(0, df['Top Score'].max()+2), align='left')
plt.title('Distribution of Top Match Scores per Requirement')
plt.xlabel('Top Match Score')
plt.ylabel('Count of Requirements')
plt.tight_layout()
plt.show()

# --- Plot 4: Bar chart of Top 10 most frequently matched NIST controls ---
all_ids = [cid for matches in df['Weighted Matches'] for cid, _ in matches]
top10 = Counter(all_ids).most_common(10)
ids, freqs = zip(*top10)
plt.figure()
plt.bar(ids, freqs)
plt.xticks(rotation=45, ha='right')
plt.title('Top 10 Most Frequently Matched NIST Controls')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# --- Plot 5: Boxplot of top match score by Control Family ---
# (assumes you have also the NIS2 dataset with family column)
df_nis2 = pd.read_csv("nis2_requirements_svo_with_family.csv")
df_nis2['Top Score'] = df['Top Score']
plt.figure(figsize=(8,5))
df_nis2.boxplot(column='Top Score', by='Control Family', rot=45)
plt.title('Top Match Score by Control Family')
plt.suptitle('')
plt.ylabel('Top Score')
plt.tight_layout()
plt.show()

Baseline diagnostic note:
- the plots below describe the weighted SVO experiment only
- they must not be presented as validation of the final semantic pipeline without manual benchmarking


In [ ]:
import pandas as pd
import ast

# 1. Carica il file con i Weighted Matches
df = pd.read_csv("nis2_nist_weighted_svo_matching.csv")
df['Weighted Matches'] = df['Weighted Matches'].apply(ast.literal_eval)

# 2. “Srotola” la lista di matches in righe
rows = []
for _, r in df.iterrows():
    for ctl, score in r['Weighted Matches']:
        rows.append({
            "Req ID": r["Req ID"],
            "Control Family": r["Control Family"],
            "Control ID": ctl,
            "Score": score
        })
flat_df = pd.DataFrame(rows)

# 3. Salva e scarica
flat_df.to_csv("nis2_nist_flat_matching.csv", index=False)

Note:
- semantic similarity is the final matching strategy and is implemented in Phase 3
- keep SVO results as comparison material, not as the primary thesis outcome
